# AutoRA Workflow

Generated by AutoRA Workflow Editor on 2026-08-28T19:57:57.606Z

## 1. Install dependencies

In [ ]:
%pip install autora-theorist-darts==1.1.0 autora-synthetic==2.2.0 git+https://github.com/AutoResearch/autora-experimentalist-lhs.git@531e728e8933650572d9084141802c36b40ddcd5

## 2. Imports

In [ ]:
from autora.state import on_state, Delta, estimator_on_state, StandardState
from autora.variable import VariableCollection, IV, DV
from autora.experimentalist.lhs import pool as latin_hypercube_pooler, sample as latin_hypercube_sampler
from autora.experiment_runner.synthetic.abstract.lmm import lmm_experiment
from autora.theorist.darts.regressor import DARTSRegressor

import pandas as pd
import numpy as np

## 3. Component definitions

In [ ]:
# Latin Hypercube Pooler
@on_state()
def latin_hypercube_pooler_on_state(variables: VariableCollection) -> Delta:
    return Delta(conditions=latin_hypercube_pooler(variables, num_samples=1))

In [ ]:
# Latin Hypercube Sampler
@on_state()
def latin_hypercube_sampler_on_state(conditions: pd.DataFrame, experiment_data: pd.DataFrame, variables: VariableCollection, num_samples: int = 1) -> Delta:
    reference_conditions = experiment_data[[v.name for v in variables.independent_variables]] if experiment_data is not None else conditions.iloc[0:0]
    return Delta(conditions=latin_hypercube_sampler(conditions=conditions, reference_conditions=reference_conditions, num_samples=num_samples))

In [ ]:
# Linear Mixed Model Experiment (Synthetic, Abstract)
runner = lmm_experiment(
    formula="rt ~ 1 + x1", fixed_effects={'Intercept': 0., 'x1': 2.},
    # TODO: adjust the variable names and ranges below for your experiment
    X=[IV(name="x1", value_range=(-10, 10))])

@on_state()
def linear_mixed_model_experiment_on_state(conditions: pd.DataFrame) -> Delta:
    return Delta(experiment_data=runner.run(conditions=conditions))

In [ ]:
# DARTS Regressor
darts_regressor_on_state = estimator_on_state(DARTSRegressor(batch_size=64, num_graph_nodes=2, output_type="real", classifier_weight_decay=0.01, darts_type="original", param_updates_per_epoch=10, param_updates_for_sampled_model=100, param_learning_rate_max=0.025, param_learning_rate_min=0.01, param_momentum=0.9, arch_updates_per_epoch=1, arch_learning_rate_max=0.003, arch_weight_decay=0.0001, arch_weight_decay_df=0.0003, arch_weight_decay_base=0, arch_momentum=0.9, fair_darts_loss_weight=1, max_epochs=10, grad_clip=5, primitives=["none", "add", "subtract", "linear", "linear_logistic", "linear_relu"], train_classifier_coefficients=False, train_classifier_bias=False, sampling_strategy="max"))

## 4. Run the workflow

In [ ]:
# Variables are governed by the experiment runner defined above
assert runner.variables is not None
variables = runner.variables

# Initialize state
state = StandardState(variables=variables)

# Experiment loop (1 cycles)
for cycle_0 in range(1):
    print(f'Cycle {cycle_0}')

    # Latin Hypercube Pooler
    state = latin_hypercube_pooler_on_state(state)

    # Latin Hypercube Sampler
    state = latin_hypercube_sampler_on_state(state, num_samples=1)

    # Linear Mixed Model Experiment (Synthetic, Abstract)
    state = linear_mixed_model_experiment_on_state(state)

    # DARTS Regressor
    state = darts_regressor_on_state(state)


print("Workflow completed!")
state